In [ ]:
#https://bioconductor.org/packages/release/bioc/vignettes/MutationalPatterns/inst/doc/Introduction_to_MutationalPatterns.html#:~:text=Mutational%20signatures%20are%20thought%20to,negative%20matrix%20factorization%20(NMF).

In [ ]:
library(future)
# cores aka "workers", each gets 16 GB RAM allocated less ~2 GB per core for overhead 
plan("multicore")
num_workers <- 10#nbrOfWorkers()
options(future.globals.maxSize = 48000 *1024^2)#14000 * num_workers * 1024^2) 

In [ ]:
library(ggplot2)
library(ggpubr)
library(rstatix)
library(data.table)
library(MutationalPatterns)

In [ ]:
ref_genome <- "BSgenome.Hsapiens.UCSC.hg38"
library(ref_genome, character.only = TRUE)

In [ ]:
vcf_files <- list.files('placental/maf2vcf/', full.names = TRUE)

In [ ]:
sample_names <- c('P1017',
'P1018',
'P1028',
'P1031',
'P1034',
'P1043',
'P1045',
'P1080',
'P1105',
'P1120',
'P1129',
'P1155',
'P1160',
'P1172',
'P1178',
'P1205',
'P1213',
'P1231',
'P1232',
'P1234',
'P1256',
'P1257',
'P1270',
'P1297',
'P1298',
'P1330',
'P1332',
'P1371',
'P1386',
'P1389',
'P1401',
'P1421',
'P1430',
'P1434',
'P1437',
'P1438',
'P1442',
'P1447',
'P1448',
'P1449',
'P1458',
'P1508',
'P1548',
'P1565',
'P1580',
'P1584',
'P1614',
'P1620',
'P1622',
'P1632',
'P1634',
'P1635',
'P1636',
'P1645',
'P1653',
'P1666',
'P1693',
'P1702',
'P1719',
'P1730',
'P1737',
'P1745',
'P1787',
'P1831',
'P1866',
'P1870',
'P1881',
'P1912',
'P1932',
'P1938',
'P1940',
'P1943',
'P1950',
'P1956',
'P538',
'P550',
'P571',
'P582',
'P600',
'P604',
'P606',
'P611',
'P629',
'P641',
'P664',
'P687',
'P709',
'P718',
'P753',
'P760',
'P768',
'P814',
'P815',
'P818',
'P824',
'P826',
'P829',
'P855',
'P857',
'P864',
'P869',
'P873',
'P882',
'P918',
'P931',
'P942',
'P950',
'P972',
'P976',
'P979',
'P980',
'P990',
                 'SampleP1_1154',
'SampleP1_1651',
'SampleP1_1689',
'SampleP1_1748',
'SampleP1_1973',
'SampleP1_2013',
'SampleP1_2052',
'SampleP1_2139',
'SampleP1_2263',
'SampleP1_2326',
'SampleP1_2591',
'SampleP1_2672',
'SampleP1_2699',
'SampleP1_2703',
'SampleP1_2778',
'SampleP1_2858',
'SampleP1_2969',
'SampleP1_830',
'SampleP3_1536',
'SampleP3_1955',
'SampleP6_1336',
'SampleP7_1162',
'SampleP7_1362',
'SampleP7_1382',
'SampleP7_1482',
'SampleP7_1496',
'SampleP7_1571',
'SampleP7_1733',
'SampleP7_1853',
'SampleP7_1897',
'SampleP7_1907',
'SampleP7_2109',
'SampleP7_2190',
'SampleP7_2371',
'SampleP7_2478',
'SampleP7_2511',
'SampleP7_2553',
'SampleP7_2566',
'SampleP7_2634',
'SampleP7_2836',
'SampleP7_791',
'SampleP7_949',
'SampleP7_988',
'SampleP8_1789',
'SampleP9_1181',
'SampleP9_1511',
'SampleP9_1672',
'SampleP9_1796',
'SampleP9_1805')

In [ ]:
grl <- read_vcfs_as_granges(vcf_files, sample_names, ref_genome)
save(grl, file="grl_112_somatic_discovery_and_validation.rdata")

In [ ]:
load(file="grl_112_somatic_discovery_and_validation.rdata")

In [ ]:
meta<- read.csv("Disc_Val_metadata_combined_short_20260402.csv", stringsAsFactors=FALSE)

In [ ]:
head(meta)

In [ ]:
#### need to merge meata

In [ ]:
meta$Sig_Group <- paste(meta$PE_onset_uniform, meta$Path_MVM_binary, sep="_")
meta$PE_MVM_Cohort <- paste(meta$Sig_Group, meta$variant_cohort, sep="_")


In [ ]:
#CHECK that META is in same order as sample names for combined
identical(meta$Sample,sample_names)

In [ ]:
#define metadata
PE_MVM <- meta$Sig_Group
PE_MVM_Cohort <- meta$PE_MVM_Cohort
PE<- meta$PE_binary

PE_onset <- meta$PE_onset_uniform

lesion <- meta$Path_binary

MVM <- meta$Path_MVM_binary

SGA <- meta$SGA_binary

In [ ]:
muts <- mutations_from_vcf(grl[[1]])
head(muts, 12)

In [ ]:
types <- mut_type(grl[[1]])
head(types, 12)

In [ ]:
context <- mut_context(grl[[1]], ref_genome)
head(context, 12)

In [ ]:
type_context <- type_context(grl[[1]], ref_genome)
lapply(type_context, head, 12)


In [ ]:
type_occurrences <- mut_type_occurrences(grl, ref_genome)
type_occurrences

In [ ]:
p1 <- plot_spectrum(type_occurrences)
p1

In [ ]:
p2 <- plot_spectrum(type_occurrences, CT = TRUE)
p2

In [ ]:
p3 <- plot_spectrum(type_occurrences, CT = TRUE, 
                    indv_points = TRUE, legend = FALSE)
p3

In [ ]:
library("gridExtra")
grid.arrange(p1, p2, p3, ncol = 3, widths = c(3, 3, 1.75))

In [ ]:
mut_mat <- mut_matrix(vcf_list = grl, ref_genome = ref_genome)
head(mut_mat)

In [ ]:
plot_96_profile(mut_mat[, c(1, 7)])


In [ ]:
mut_mat_ext_context <- mut_matrix(grl, ref_genome, extension = 2)
head(mut_mat_ext_context)

In [ ]:
plot_profile_heatmap(mut_mat_ext_context)


In [ ]:
mut_mat <- mut_mat + 0.0001


In [ ]:
mut_mat <- mut_mat + 0.0001
mut_mat <- mut_mat + 0.0001
library("NMF")
estimate <- nmf(mut_mat, rank = 2:6, method = "brunet", 
                nrun = 1000, seed = 123456, .opt = "v-p")
save(estimate, file="Discovery_Validation_estimate_20260304.rdata")

In [ ]:
load(file="Discovery_Validation_estimate_20260304.rdata")

In [ ]:
plot(estimate)


In [ ]:
pdf("Discovery_Validation_mutational_signatures_NMF_estimate.pdf")
p<-plot(estimate)
print(p)
dev.off()

In [ ]:
nmf_res <- extract_signatures(mut_mat, rank = 4, nrun = 1000, single_core = FALSE)
save(nmf_res, file="Discovery_Validation_nmf_res_20260225_rannk4.rdata")
write.csv(nmf_res$contribution, "Discovery_Validation_NMFres_signatures_by_sample_four_sigs_20260302.csv")

In [ ]:
load(file="Discovery_Validation_nmf_res_20260225_rannk4.rdata")

In [ ]:
getwd()

In [ ]:
head(nmf_res$signatures)

In [ ]:
write.csv(nmf_res$signatures, "Discovery_Validation_NMFres_signatures_for_cosine_sim_four_20260901.csv")

In [ ]:
pdf("Discovery_Validation_Profiles_Signatures_four.pdf")
p<-plot_96_profile(nmf_res$signatures, condensed = TRUE)
print(p)
p<-plot_96_profile(nmf_res$signatures, condensed = FALSE)
print(p)
dev.off()

In [ ]:
signatures = get_known_signatures( genome='GRCh38')
cos_sim_samples_signatures <- cos_sim_matrix(nmf_res$signatures, signatures)
cos_sim_samples_signatures
write.csv(cos_sim_samples_signatures, "Discovery_Validation_cos_sim_samples_signatures_20260303_fourv2.csv")

In [ ]:
cos_sim_samples_signatures_sub <- cos_sim_samples_signatures[, apply(cos_sim_samples_signatures, 2, min) > .4]
head(cos_sim_samples_signatures_sub)

In [ ]:
# Install and load the package
#install.packages("pheatmap")
library(pheatmap)
library(RColorBrewer)

# Generate the pretty heatmap
plot1 <- pheatmap(t(cos_sim_samples_signatures_sub),color = colorRampPalette(rev(brewer.pal(n = 9, name = "RdYlBu")))(1000),
                  cluster_cols = FALSE,cluster_rows = TRUE)
print(plot1)

pdf("Mutational_signatures_discovery_validation_cosine_similarity_heatmap_final_20261021_v2_sub_notpooled.pdf")
print(plot1)
dev.off()

In [ ]:
pooled_mut_mat <- pool_mut_mat(mut_mat, grouping=PE_MVM_Cohort)
head(pooled_mut_mat)

nmf_res_pooled <- extract_signatures(pooled_mut_mat, rank = 4, nrun = 1000, single_core = FALSE)
signatures = get_known_signatures()
nmf_res_pooled <- rename_nmf_signatures(nmf_res_pooled, signatures, cutoff = 0.99)
write.csv(nmf_res_pooled$contribution, "NMF_res_pooled_PE_MVM_Cohort_somatic_Discovery_20260409v2.csv")
colnames(nmf_res_pooled$signatures)
plot_96_profile(nmf_res_pooled$signatures, condensed = TRUE)


p<-plot_contribution_heatmap(nmf_res_pooled$contribution, 
                          cluster_samples = TRUE, 
                          cluster_sigs = TRUE)

print(p)

pdf("Mutational_signatures_nmf_res_pooled_PE_MVM_somatic_Discovery_Validation_20260402v2.pdf")
print(p)
dev.off()

In [ ]:
head(sigs_meta_disc)

In [ ]:
samps <- sigs_meta_disc$Sample

In [ ]:
head(mut_mat)

In [ ]:
mut_mat_discovery <- mut_mat[,samps]
dim(mut_mat)
dim(mut_mat_discovery)

In [ ]:
PE_MVM_Cohort_disc<-sigs_meta_disc$PE_MVM_Cohort

In [ ]:
pooled_mut_mat <- pool_mut_mat(mut_mat_discovery, grouping=PE_MVM_Cohort_disc)
head(pooled_mut_mat)

nmf_res_pooled <- extract_signatures(pooled_mut_mat, rank = 4, nrun = 1000, single_core = FALSE)
signatures = get_known_signatures()
nmf_res_pooled <- rename_nmf_signatures(nmf_res_pooled, signatures, cutoff = 0.99)
write.csv(nmf_res_pooled$contribution, "NMF_res_pooled_PE_MVM_Cohort_somatic_Discovery_only_20260409v2.csv")
colnames(nmf_res_pooled$signatures)
plot_96_profile(nmf_res_pooled$signatures, condensed = TRUE)


p<-plot_contribution_heatmap(nmf_res_pooled$contribution, 
                          cluster_samples = TRUE, 
                          cluster_sigs = TRUE)

print(p)

pdf("Mutational_signatures_nmf_res_pooled_PE_MVM_somatic_Discovery_only_20260409v2.pdf")
print(p)
dev.off()

In [ ]:
PE_MVM_Cohort_disc<-sigs_meta_disc$PE_onset_uniform
pooled_mut_mat <- pool_mut_mat(mut_mat_discovery, grouping=PE_MVM_Cohort_disc)
head(pooled_mut_mat)

nmf_res_pooled <- extract_signatures(pooled_mut_mat, rank = 4, nrun = 1000, single_core = FALSE)
signatures = get_known_signatures()
nmf_res_pooled <- rename_nmf_signatures(nmf_res_pooled, signatures, cutoff = 0.99)
write.csv(nmf_res_pooled$contribution, "NMF_res_pooled_PEonset_somatic_Discovery_only_20260409v2.csv")
colnames(nmf_res_pooled$signatures)
plot_96_profile(nmf_res_pooled$signatures, condensed = TRUE)


p<-plot_contribution_heatmap(nmf_res_pooled$contribution, 
                          cluster_samples = TRUE, 
                          cluster_sigs = TRUE)

print(p)

pdf("Mutational_signatures_nmf_res_pooled_PE_onset_somatic_Discovery_only_20260409v2.pdf")
print(p)
dev.off()

In [ ]:
p<-plot_96_profile(nmf_res_pooled$signatures, condensed = TRUE)
pdf("Mutational_signatures_nmf_res_pooled_PE_onset_somatic_Discovery_only_20260409_plot_96_profilev2.pdf")
print(p)
dev.off()

In [ ]:
nmf_res_pooled$contribution

In [ ]:
p<-plot_contribution_heatmap(nmf_res_pooled$contribution,  sample_order = c("PE_EO","PE_LO","Normal_EO","Normal_LO"), 
                             plot_values=FALSE,
 
                          cluster_samples = FALSE, 
                          cluster_sigs = FALSE)

print(p)

pdf("Mutational_signatures_nmf_res_pooled_PE_onset_somatic_Discovery_only_20260409_reorderedv2.pdf")
print(p)
dev.off()

In [ ]:
nmf_res_pooled$contribution

In [ ]:
signatures = get_known_signatures( genome='GRCh38')
cos_sim_samples_signatures <- cos_sim_matrix(nmf_res_pooled$signatures, signatures)
cos_sim_samples_signatures
write.csv(cos_sim_samples_signatures, "Discovery_Validation_pooled_cos_sim_samples_signatures_20261021_four_v2.csv")

In [ ]:
cos_sim_samples_signatures_pooled_sub <- cos_sim_samples_signatures[, apply(cos_sim_samples_signatures, 2, min) > .49]
head(cos_sim_samples_signatures_pooled_sub)

In [ ]:
# Install and load the package
#install.packages("pheatmap")
library(pheatmap)
library(RColorBrewer)

# Generate the pretty heatmap
plot1 <- pheatmap(t(cos_sim_samples_signatures_pooled_sub),color = colorRampPalette(rev(brewer.pal(n = 9, name = "RdYlBu")))(1000),
                  cluster_cols = FALSE,cluster_rows = TRUE)
print(plot1)

pdf("Mutational_signatures_discovery_validation_cosine_similarity_heatmap_final_20261021_v2_sub.pdf")
print(plot1)
dev.off()